# 📈 CerebroSpheroid V5: Live Loss Trajectory & Regression Monitor

This notebook tracks training progress in real-time across **Cloud TPU, Kaggle 2xT4, and Colab** nodes via the `bioflo.in` checkpoint hub.

### What This Monitor Tracks:
- **True Loss vs. Batch Variance**: Computes running Exponential Moving Average (EMA) and optional deterministic held-out validation loss.
- **Instant Regression Detection**: Alerts on loss jumps exceeding the configured threshold (e.g., `+0.15`) or consecutive upward drift.
- **Milestone Horizon Forecasting**: Uses empirical scaling dynamics to project steps remaining until sub-3.0 and sub-2.5 loss milestones.
- **In-Place Live Dashboard**: Updates interactive matplotlib plots dynamically in Colab without cluttering the output.

## 1. Setup & Repository Synchronization

In [ ]:
#@title 🛠️ Step 1: Clone CerebroSpheroid V5 & Install Dependencies
import os, subprocess, sys

REPO_URL = "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git"
REPO_DIR = "/content/cerebro_spheroid_v5"

if not os.path.exists(REPO_DIR):
    print(f"📦 Cloning CerebroSpheroid V5 from {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print("🔄 Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

%cd {REPO_DIR}
!pip install -q torch transformers datasets matplotlib
print("✓ Repository ready & dependencies verified!")


## 2. Configure Loss Monitor Parameters

In [ ]:
#@title ⚙️ Step 2: Configure Live Tracking Options
MODEL_PRESET = "v5_small"  #@param ["v5_small", "v5_compact", "v5_base"]
WATCH_MODE = True  #@param {type:"boolean"}
POLL_INTERVAL_SECONDS = 60  #@param {type:"integer"}
EVALUATE_HELD_OUT_LOSS = False  #@param {type:"boolean"}
REGRESSION_SPIKE_THRESHOLD = 0.15  #@param {type:"number"}
SAVE_TO_WEBSITE = True  #@param {type:"boolean"}

print("⚙️ Monitor Parameters Confirmed:")
print(f"  • Model Preset         : {MODEL_PRESET}")
print(f"  • Continuous Watch     : {WATCH_MODE} (polling every {POLL_INTERVAL_SECONDS}s)")
print(f"  • Held-Out Validation  : {EVALUATE_HELD_OUT_LOSS}")
print(f"  • Regression Threshold : +{REGRESSION_SPIKE_THRESHOLD:.2f} loss jump")
print(f"  • Website Data Sync   : {SAVE_TO_WEBSITE} (https://bioflo.in/api/cerebro/data/)\n")


## 3. Launch Live Regression Tracker Dashboard

In [ ]:
#@title 🚀 Step 3: Run In-Place Live Trajectory Dashboard
from scripts.track_loss_regression import run_loss_tracker

records, analysis = run_loss_tracker(
    model=MODEL_PRESET,
    watch=WATCH_MODE,
    interval_sec=POLL_INTERVAL_SECONDS,
    eval_model=EVALUATE_HELD_OUT_LOSS,
    regression_threshold=REGRESSION_SPIKE_THRESHOLD,
    save_to_cloud=SAVE_TO_WEBSITE,
)


## 4. Analytical Summary & Milestone Horizons

In [ ]:
#@title 📊 Step 4: Display Historical Table & Milestone Projections
import json, os

hist_file = f"loss_history_{MODEL_PRESET}.json"
if os.path.exists(hist_file):
    with open(hist_file, "r", encoding="utf-8") as f:
        history = json.load(f)
    print(f"{'Step':<8} | {'Reported Loss':<14} | {'EMA Loss':<10} | {'Eval Loss':<10} | {'Δ Loss':<8} | {'Status':<22} | {'Timestamp'}")
    print("-" * 98)
    for r in history[-15:]:  # Show last 15 checkpoints
        eval_s = f"{r.get('eval_loss'):.4f}" if r.get('eval_loss') is not None else "N/A"
        ema_s = f"{r.get('ema_loss'):.4f}" if r.get('ema_loss') is not None else "N/A"
        delta_s = f"{r.get('delta'):+.4f}" if r.get('delta') is not None else "0.0000"
        print(f"{r['step']:<8d} | {r['loss']:<14.4f} | {ema_s:<10} | {eval_s:<10} | {delta_s:<8} | {r['status']:<22} | {r.get('timestamp','')[:19]}")
else:
    print("ℹ️ No local history file recorded yet. Run Step 3 first to populate.")
